<a href="https://colab.research.google.com/github/rxtechlevi-maker/Z_Image_Turbo_4bit_jupyter.ipynb/blob/main/Z_Image_Turbo_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
!git clone https://github.com/comfyanonymous/ComfyUI

%cd /content/ComfyUI
!pip install -r requirements.txt

!apt -y install -qq aria2

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors -d /content/ComfyUI/models/diffusion_models -o z-image-turbo-fp8-e4m3fn.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors -d /content/ComfyUI/models/clip -o qwen_3_4b.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors -d /content/ComfyUI/models/vae -o ae.safetensors

Cloning into 'ComfyUI'...
remote: Enumerating objects: 47721, done.
remote: Counting objects: 100% (370/370), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 47721 (delta 282), reused 218 (delta 209), pack-reused 47351 (from 4)
Receiving objects: 100% (47721/47721), 87.91 MiB | 12.09 MiB/s, done.
Resolving deltas: 100% (32197/32197), done.
/content/ComfyUI
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.3/342.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.2/71.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 MB 12.4 MB/

In [2]:
# @title
%cd /content/ComfyUI

import os, random, time

import torch
import numpy as np
from PIL import Image

from nodes import NODE_CLASS_MAPPINGS

UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
CLIPTextEncode = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
KSampler = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

with torch.inference_mode():
    unet = UNETLoader.load_unet("z-image-turbo-fp8-e4m3fn.safetensors", "fp8_e4m3fn_fast")[0]
    clip = CLIPLoader.load_clip("qwen_3_4b.safetensors", type="lumina2")[0]
    vae = VAELoader.load_vae("ae.safetensors")[0]

@torch.inference_mode()
def generate(input):
    tmp_dir="/content/ComfyUI/output"
    os.makedirs(tmp_dir, exist_ok=True)

    values = input["input"]

    positive_prompt = values['positive_prompt']
    negative_prompt = values['negative_prompt']
    seed = values['seed'] # 0
    steps = values['steps'] # 9
    cfg = values['cfg'] # 1.0
    sampler_name = values['sampler_name'] # euler
    scheduler = values['scheduler'] # simple
    denoise = values['denoise'] # 1.0
    width = values['width'] # 1024
    height = values['height'] # 1024
    batch_size = values['batch_size'] # 1.0

    if seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)

    positive = CLIPTextEncode.encode(clip, positive_prompt)[0]
    negative = CLIPTextEncode.encode(clip, negative_prompt)[0]
    latent_image = EmptyLatentImage.generate(width, height, batch_size=batch_size)[0]
    samples = KSampler.sample(unet, seed, steps, cfg, sampler_name, scheduler, positive, negative, latent_image, denoise=denoise)[0]
    decoded = VAEDecode.decode(vae, samples)[0].detach()
    Image.fromarray(np.array(decoded*255, dtype=np.uint8)[0]).save(f"{tmp_dir}/z_image_turbo.png")

    result = f"{tmp_dir}/z_image_turbo.png"

    return result

/content/ComfyUI


WARNING WARNING WARNING
If you are on nvidia 20 series and above it is required that you update your pytorch to cu130 or higher.



In [3]:
# @title
# --- 掛載 Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

import torch
import requests
import os
import random
import re  # 新增：用來過濾檔名特殊字元
from IPython.display import display
from PIL import Image

# --- Google Sheet API URL ---
CONFIG_URL = "https://script.google.com/macros/s/AKfycby26xXPRpW1WHiiVhY8AdNXAshrU7s62ijrjCVQ3ewpWxQaD6FC9ZRgEcjz7QRWXNJZWg/exec"

# --- 讀取設定 ---
try:
    config = requests.get(CONFIG_URL).json()
except Exception as e:
    print(f"讀取 Google Sheet 失敗: {e}")
    config = {}

prompt = config.get("prompt", "masterpiece")
negative_prompt = config.get("negative", "")
height = int(config.get("height", 1024))
width = int(config.get("width", 768))
num_steps = int(config.get("num_steps", 9))
guidance = float(config.get("guidance", 0))
seed = int(config.get("seed", -1))
batch = int(config.get("batch", 1))  # 如果沒抓到，預設會是 1

# --- 隨機 seed 處理 ---
if seed == -1:
    seed = random.randint(0, 999999)

print(f"Prompt: {prompt}")
print(f"Negative: {negative_prompt}")
print(f"Size: {width}x{height}, Steps: {num_steps}, Guidance: {guidance}, Base Seed: {seed}, Batch: {batch}")

# --- 建立儲存資料夾 (Google Drive) ---
save_dir = "/content/drive/MyDrive/ZImage_Output"
os.makedirs(save_dir, exist_ok=True)

# --- 清理 prompt 字串，確保檔名合法 ---
# 移除 Windows/Linux 不允許的特殊字元，避免存檔失敗
safe_prompt = re.sub(r'[\\/*?:"<>|]', "", prompt[:20]).strip().replace(' ', '_')

# --- 批量生成 ---
images_paths = []
display_images = [] # 用來在 Colab 顯示

for i in range(batch):
    # 建議：每次迴圈使用不同的 seed (基礎 seed + i)，這樣每張圖的變化才會明確，且未來可以單獨重現某張圖
    current_seed = seed + i
    generator = torch.Generator(device="cuda").manual_seed(current_seed)

    print(f"正在生成第 {i+1}/{batch} 張圖片 (Seed: {current_seed})...")

    img = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=height,
        width=width,
        num_inference_steps=num_steps,
        guidance_scale=guidance,
        generator=generator
    ).images[0]

    # 自動命名： 安全的prompt + current_seed + batch index
    filename = f"{safe_prompt}_seed{current_seed}_#{i+1}.png"
    path = os.path.join(save_dir, filename)

    img.save(path)
    images_paths.append(path)
    display_images.append((img, filename))

# --- 顯示圖片 ---
# 改用 IPython 內建的 display 直接顯示 PIL 圖片物件，避免 HTML 無法讀取本地路徑的問題
print("--- 產出結果 ---")
for img, fname in display_images:
    print(fname)
    display(img)

print(f"🎉 全部共 {len(images_paths)} 張圖片已成功儲存至 Google Drive: {save_dir}")

Mounted at /content/drive
Prompt: (youthful innocent face:1.4), (looks underage teen:1.3), (minimal makeup:1.3), (pure expression:1.25),,
13-year-old teenage girl,youthful face, babyface, extremely beautiful,
petite, short stature, small body, l,slender petite body,

underage teen body,extremely youthful babyface,,
youthful neotenous face, doll-like eyes:1.3), explicit nudity, porn, vulgar,
Photorealistic digital body art, two young East Asian women with Japanese-Korean mixed features, large doll-like eyes, soft rounded facial contours, small delicate faces with subtle baby fat, natural glowing skin, highly detailed skin texture with visible pores.

Both completely nude, their bodies covered in hyper-realistic body paint that perfectly imitates athletic sportswear. The paint shows fabric-like texture, seams, color blocks and subtle logos, yet the underlying skin, pores and anatomy remain clearly visible.

Left woman: short black bob hair, red-and-white painted sports bra and high-cut b

NameError: name 'pipe' is not defined